In [2]:
import pandas as pd
from pathlib import Path
import numpy as np
import json
ROOT = Path.cwd().parent
motor='DATA_MODEL_Dev/Motor_data_models.csv'
data_split='DATA_RAW_MODELS/Data_split.json'

# DATA SPLIT

In [3]:
with open(ROOT/data_split, "r", encoding="utf-8") as archivo:
    data_subjects = json.load(archivo)

subjects_train=data_subjects['Train_80']
subjects_test=data_subjects['Test_20']

In [4]:
X_train=pd.read_csv(ROOT/motor)
X_train=X_train[X_train['subject_visit'].isin(subjects_train)]
X_train.drop(columns=['subject_visit','UPDRS_III_ProgressionType'], inplace=True)

X_test=pd.read_csv(ROOT/motor)
X_test=X_test[X_test['subject_visit'].isin(subjects_test)]
X_test.drop(columns=['subject_visit','UPDRS_III_ProgressionType'], inplace=True)

y_train=pd.read_csv(ROOT/motor)
y_train=y_train[y_train['subject_visit'].isin(subjects_train)]
y_train=y_train['UPDRS_III_ProgressionType']
y_test=pd.read_csv(ROOT/motor)
y_test=y_test[y_test['subject_visit'].isin(subjects_test)]
y_test=y_test['UPDRS_III_ProgressionType']


## DATA TRAIN

In [5]:
X_train.shape, y_train.shape

((728, 9), (728,))

In [6]:
X_train.head()

,SCHWAB & ENGLAND ADL,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,Does participant have DBS,MDS-UPDRS Part III Total Score,3.21 HOEHN AND YAHR STAGE,MDS-UPDRS Part IV Total Score,DBS_Transition_Visit,DBS_Post_Transition
0,90,6,5.0,0,35,2,0.0,0,0.0
1,85,9,10.0,0,53,3,0.0,0,0.0
2,80,11,16.0,0,18,2,7.0,0,0.0
3,90,8,8.0,0,34,2,0.0,0,0.0
4,85,4,9.0,0,16,2,0.0,0,0.0


In [7]:
y_train.head()

0   -1
1   -1
2    1
3    0
4    0
Name: UPDRS_III_ProgressionType, dtype: int64

## DATA TEST

In [8]:
X_test.shape, y_test.shape

((183, 9), (183,))

In [9]:
X_test.head()

,SCHWAB & ENGLAND ADL,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,Does participant have DBS,MDS-UPDRS Part III Total Score,3.21 HOEHN AND YAHR STAGE,MDS-UPDRS Part IV Total Score,DBS_Transition_Visit,DBS_Post_Transition
7,90,5,5.0,0,18,2,5.0,0,0.0
10,90,4,1.0,0,15,2,4.0,0,0.0
14,80,2,7.0,0,13,2,1.0,0,0.0
18,90,2,3.0,0,15,2,0.0,0,0.0
19,80,2,4.0,0,9,2,0.0,0,0.0


In [10]:
y_test.head()

7     0
10   -1
14   -1
18   -1
19    0
Name: UPDRS_III_ProgressionType, dtype: int64

# Feature Selection No Model Univariate

In [11]:
from sklearn.feature_selection import GenericUnivariateSelect
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif
from sklearn.feature_selection import VarianceThreshold

In order to perform univariate feature selection we must summarize the methodology and all the methods that we will study:
1. Variance treshold: Fast/Easy it may not capture the all picture. 
2. Using the GenericUnivarianceSelect method we can apply SelectKBest/SelectPercentile and the statistical function associated.

## VarianceThreshold

In [12]:
threshold_range = np.arange(0, 2, 0.1).tolist()

for tresh in threshold_range:
    selector = VarianceThreshold(threshold=tresh)
    X_train_reduced = selector.fit_transform(X_train)
    n_features = X_train_reduced.shape[1]
    print(f'Threshold: {tresh:.2}, Number of features retained: {n_features}')
    # Columnas eliminadas
    eliminadas = X_train.columns[~selector.get_support()]
    print("Deleted:", eliminadas.tolist())

    # Columnas conservadas
    conservadas = X_train.columns[selector.get_support()]
    print("Conserved:", conservadas.tolist())
    print("---------------------------------------------------")



Threshold: 0.0, Number of features retained: 9
Deleted: []
Conserved: ['SCHWAB & ENGLAND ADL', 'MDS-UPDRS Part I (Patient Questionnaire) Total Score', 'MDS-UPDRS Part II Total Score', 'Does participant have DBS', 'MDS-UPDRS Part III Total Score', '3.21 HOEHN AND YAHR STAGE', 'MDS-UPDRS Part IV Total Score', 'DBS_Transition_Visit', 'DBS_Post_Transition']
---------------------------------------------------
Threshold: 0.1, Number of features retained: 7
Deleted: ['Does participant have DBS', 'DBS_Transition_Visit']
Conserved: ['SCHWAB & ENGLAND ADL', 'MDS-UPDRS Part I (Patient Questionnaire) Total Score', 'MDS-UPDRS Part II Total Score', 'MDS-UPDRS Part III Total Score', '3.21 HOEHN AND YAHR STAGE', 'MDS-UPDRS Part IV Total Score', 'DBS_Post_Transition']
---------------------------------------------------
Threshold: 0.2, Number of features retained: 6
Deleted: ['Does participant have DBS', 'DBS_Transition_Visit', 'DBS_Post_Transition']
Conserved: ['SCHWAB & ENGLAND ADL', 'MDS-UPDRS Part I

This method is not usefull due to some features that have low variance are related to not frequent events

## Generic Univariate Select

In [13]:
mode_list = ['percentile', 'k_best']
score_funcs = [f_classif, mutual_info_classif, chi2]
k_list = range(5, 10)
percentile_list = range(10, 100, 10)

results = []

for mode in mode_list:
    for score in score_funcs:

        if mode == 'k_best':
            for k in k_list:
                
                transformer = GenericUnivariateSelect(
                    score_func=score,
                    mode=mode,
                    param=k
                )

                X_new = transformer.fit_transform(X_train, y_train)

                selected_features = X_train.columns[transformer.get_support()]
                delete_features = X_train.columns[~transformer.get_support()]

                results.append({
                    "mode": mode,
                    "score_func": score.__name__,
                    "param": k,
                    "n_features": X_new.shape[1],
                    "features Added": list(selected_features),
                    "features Deleted": list(delete_features)
                })

        elif mode == 'percentile':
            for p in percentile_list:
                
                transformer = GenericUnivariateSelect(
                    score_func=score,
                    mode=mode,
                    param=p
                )

                X_new = transformer.fit_transform(X_train, y_train)

                selected_features = X_train.columns[transformer.get_support()]
                delete_features = X_train.columns[~transformer.get_support()]

                results.append({
                    "mode": mode,
                    "score_func": score.__name__,
                    "param": p,
                    "n_features": X_new.shape[1],
                    "features Added": list(selected_features),
                    "features Deleted": list(delete_features)
                })

df_results = pd.DataFrame(results)
df_results.to_csv(ROOT/'DATA_MODEL_Dev/Motor_feature_selection_results.csv', index=False)
df_results.head()


,mode,score_func,param,n_features,features Added,features Deleted
0,percentile,f_classif,10,1,[MDS-UPDRS Part III Total Score],"[SCHWAB & ENGLAND ADL, MDS-UPDRS Part I (Patie..."
1,percentile,f_classif,20,2,"[MDS-UPDRS Part III Total Score, 3.21 HOEHN AN...","[SCHWAB & ENGLAND ADL, MDS-UPDRS Part I (Patie..."
2,percentile,f_classif,30,3,"[MDS-UPDRS Part II Total Score, MDS-UPDRS Part...","[SCHWAB & ENGLAND ADL, MDS-UPDRS Part I (Patie..."
3,percentile,f_classif,40,4,"[SCHWAB & ENGLAND ADL, MDS-UPDRS Part II Total...",[MDS-UPDRS Part I (Patient Questionnaire) Tota...
4,percentile,f_classif,50,4,"[SCHWAB & ENGLAND ADL, MDS-UPDRS Part II Total...",[MDS-UPDRS Part I (Patient Questionnaire) Tota...


Given that the motor data only present 9 features all will be selected

# Motor Model Development
<p align="center">
    <img src="../../../figures/model_process.png" alt="Model process" width="600">
</p>





---

Linear Models  
(assume a linear decision boundary)

- Logistic Regression  
- Linear Support Vector Machine (Linear SVM)  
- SGD Classifier (logistic loss / hinge loss)

---

Non-Linear Models  
(can model complex decision boundaries)

Tree-Based Models  
- Decision Trees  
- Random Forest  
- Extra Trees (Extremely Randomized Trees)  
- Gradient Boosting  
- XGBoost  
- LightGBM  
- CatBoost  

Distance-Based Models  
- K-Nearest Neighbors (KNN)

Margin-Based Models  
- Support Vector Machines (SVM) with kernels:  
  - RBF kernel  
  - Polynomial kernel  
  - Sigmoid kernel  

Probabilistic Models  
- Multinomial Naive Bayes  
- Bernoulli Naive Bayes  
- Bayesian Classifiers  

Neural Network Models  
- Multilayer Perceptron (MLP)  
---



In [56]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn import svm
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import make_scorer, recall_score, confusion_matrix,f1_score,precision_score,balanced_accuracy_score


def specificity_weighted(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    n_classes = cm.shape[0]

    total_samples = np.sum(cm)
    specificities = []
    supports = []

    for k in range(n_classes):
        TP = cm[k, k]
        FN = np.sum(cm[k, :]) - TP
        FP = np.sum(cm[:, k]) - TP
        TN = total_samples - (TP + FN + FP)

        spec_k = TN / (TN + FP) if (TN + FP) > 0 else 0
        specificities.append(spec_k)
        supports.append(np.sum(cm[k, :]))

    return np.average(specificities, weights=supports)


def g_mean(y_true, y_pred):
    sens = recall_score(y_true, y_pred, average="macro")
    spec = specificity_weighted(y_true, y_pred)
    return np.sqrt(sens * spec)




def cv_results_to_row(results, modelo_name, parameters, sep=" ± "):

    row = {
        "Model": modelo_name,
        "Parameters": parameters
    }

    for k in results:
        if k.startswith("train_") or k.startswith("test_"):
            prefix, metric = k.split("_", 1)

            mean = results[k].mean()
            std = results[k].std()

            col_name = f"{prefix}_{metric}"
            row[col_name] = f"{mean:.4f}{sep}{std:.4f}"

    return pd.DataFrame([row])



## Dummy Classifier

In [57]:
# 1) Dummy model (baseline)
Dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)

# 2) Stratified CV
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 3) Scoring 
scoring = {
    "acc": "accuracy",
    "bal_acc": make_scorer(balanced_accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted"),
    "prec_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "rec_macro": make_scorer(recall_score, average="macro", zero_division=0),
    "specificity_weighted": make_scorer(specificity_weighted),
    "g_mean": make_scorer(g_mean),
}

# 4) Pipeline (sin scaler)
pipe_dummy = Pipeline([
    ("clf", Dummy_model)
])

# 5) Cross-validation
results_dummy = cross_validate(
    pipe_dummy,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

# 6) Convertir resultados a fila
df_dummy = cv_results_to_row(
    results_dummy,
    modelo_name="DummyClassifier",
    parameters="Most frequent (Baseline), No Scaler (Pipeline)",
    sep=" ± "
)

# 7) Mostrar resultados
df_dummy


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,DummyClassifier,"Most frequent (Baseline), No Scaler (Pipeline)",0.3956 ± 0.0020,0.3956 ± 0.0005,0.3333 ± 0.0000,0.3333 ± 0.0000,0.1890 ± 0.0007,0.1890 ± 0.0002,0.2243 ± 0.0020,0.2243 ± 0.0005,0.1319 ± 0.0007,0.1319 ± 0.0002,0.3333 ± 0.0000,0.3333 ± 0.0000,0.6044 ± 0.0020,0.6044 ± 0.0005,0.4489 ± 0.0008,0.4488 ± 0.0002


## Linear Models

### Logistic Regression

In [ ]:
# Base model
LogReg_model = LogisticRegression(class_weight="balanced", max_iter=1000)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "acc": "accuracy",
    "bal_acc": make_scorer(balanced_accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted"),
    "prec_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "rec_macro": make_scorer(recall_score, average="macro", zero_division=0),
    "specificity_weighted": make_scorer(specificity_weighted),
    "g_mean": make_scorer(g_mean),
    
}

# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", LogReg_model)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df1 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogReg_model)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_ss,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", LogReg_model)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df3 = cv_results_to_row(
    results_mm,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results = pd.concat([df1, df2, df3], ignore_index=True)
final_results


/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-lear

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LogisticRegression,"Balanced, max_iter=1000, Default Model, No Sca...",0.4464 ± 0.0425,0.4883 ± 0.0146,0.4396 ± 0.0424,0.4842 ± 0.0157,0.4378 ± 0.0444,0.4807 ± 0.0153,0.4492 ± 0.0440,0.4897 ± 0.0148,0.4448 ± 0.0461,0.4813 ± 0.0152,0.4396 ± 0.0424,0.4842 ± 0.0157,0.7222 ± 0.0232,0.7412 ± 0.0080,0.5631 ± 0.0352,0.5991 ± 0.0128
1,LogisticRegression,"Balanced, max_iter=1000, Default Model, Standa...",0.4464 ± 0.0415,0.4845 ± 0.0133,0.4384 ± 0.0412,0.4805 ± 0.0146,0.4372 ± 0.0430,0.4770 ± 0.0139,0.4492 ± 0.0427,0.4861 ± 0.0132,0.4438 ± 0.0439,0.4777 ± 0.0138,0.4384 ± 0.0412,0.4805 ± 0.0146,0.7214 ± 0.0220,0.7398 ± 0.0073,0.5620 ± 0.0338,0.5961 ± 0.0120
2,LogisticRegression,"Balanced, max_iter=1000, Default Model, MinMax...",0.4464 ± 0.0471,0.4894 ± 0.0151,0.4318 ± 0.0464,0.4795 ± 0.0147,0.4308 ± 0.0483,0.4780 ± 0.0149,0.4465 ± 0.0490,0.4900 ± 0.0151,0.4346 ± 0.0502,0.4778 ± 0.0149,0.4318 ± 0.0464,0.4795 ± 0.0147,0.7151 ± 0.0269,0.7377 ± 0.0076,0.5553 ± 0.0401,0.5947 ± 0.0121


### Linear Support Vector Machine (Linear SVM)

In [ ]:

# Modelo base
linear_svc = svm.SVC(kernel="linear")

# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", linear_svc)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="SVC_Linear",
    parameters="Default Model, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", linear_svc)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="SVC_Linear",
    parameters="Default Model, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", linear_svc)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="SVC_Linear",
    parameters="Default Model, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_svc = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_svc


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVC_Linear,"Default Model, No Standard Scaler",0.4588 ± 0.0247,0.4883 ± 0.0112,0.4058 ± 0.0211,0.4341 ± 0.0125,0.3595 ± 0.0211,0.3911 ± 0.0216,0.4017 ± 0.0221,0.4324 ± 0.0175,0.3694 ± 0.0559,0.4701 ± 0.0770,0.4058 ± 0.0211,0.4341 ± 0.0125,0.6693 ± 0.0136,0.6848 ± 0.0077,0.5210 ± 0.0187,0.5452 ± 0.0109
1,SVC_Linear,"Default Model, With Standard Scaler",0.4588 ± 0.0247,0.4883 ± 0.0136,0.4044 ± 0.0205,0.4334 ± 0.0140,0.3539 ± 0.0172,0.3885 ± 0.0211,0.3980 ± 0.0197,0.4306 ± 0.0181,0.3579 ± 0.0570,0.4781 ± 0.0800,0.4044 ± 0.0205,0.4334 ± 0.0140,0.6688 ± 0.0131,0.6844 ± 0.0093,0.5200 ± 0.0180,0.5446 ± 0.0125
2,SVC_Linear,"Default Model, With MinMax Scaler",0.4588 ± 0.0253,0.4852 ± 0.0086,0.4039 ± 0.0221,0.4285 ± 0.0066,0.3528 ± 0.0225,0.3779 ± 0.0097,0.3971 ± 0.0237,0.4223 ± 0.0068,0.3414 ± 0.0498,0.4598 ± 0.1255,0.4039 ± 0.0221,0.4285 ± 0.0066,0.6679 ± 0.0145,0.6812 ± 0.0049,0.5193 ± 0.0197,0.5403 ± 0.0060


### SGD Classifier (logistic loss / hinge loss)

In [40]:

# Modelo base
SGD_classifier = SGDClassifier(loss="hinge")

# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", SGD_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="SGD_Classifier",
    parameters="Default Model, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SGD_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="SGD_Classifier",
    parameters="Default Model, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", SGD_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="SGD_Classifier",
    parameters="Default Model, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_sgd = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_sgd

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SGD_Classifier,"Default Model, No Scaler (Pipeline)",0.4231 ± 0.0455,0.4389 ± 0.0459,0.3935 ± 0.0159,0.4104 ± 0.0180,0.3203 ± 0.0307,0.3316 ± 0.0321,0.3503 ± 0.0369,0.3610 ± 0.0428,0.3971 ± 0.1068,0.4441 ± 0.0975,0.3935 ± 0.0159,0.4104 ± 0.0180,0.6856 ± 0.0404,0.6930 ± 0.0407,0.5191 ± 0.0190,0.5331 ± 0.0235
1,SGD_Classifier,"Default Model, StandardScaler (Pipeline)",0.4381 ± 0.0607,0.4382 ± 0.0279,0.4250 ± 0.0668,0.4217 ± 0.0159,0.4135 ± 0.0738,0.4110 ± 0.0232,0.4270 ± 0.0702,0.4263 ± 0.0285,0.4260 ± 0.0704,0.4312 ± 0.0233,0.4250 ± 0.0668,0.4217 ± 0.0159,0.7060 ± 0.0408,0.7054 ± 0.0112,0.5470 ± 0.0578,0.5453 ± 0.0114
2,SGD_Classifier,"Default Model, MinMaxScaler (Pipeline)",0.4464 ± 0.0507,0.4598 ± 0.0361,0.4186 ± 0.0386,0.4395 ± 0.0354,0.3655 ± 0.0631,0.3942 ± 0.0683,0.3908 ± 0.0672,0.4138 ± 0.0651,0.4190 ± 0.0715,0.4970 ± 0.0346,0.4186 ± 0.0386,0.4395 ± 0.0354,0.7017 ± 0.0253,0.7113 ± 0.0293,0.5417 ± 0.0327,0.5589 ± 0.0328


## Non-Linear Models  

### Tree-Base Models

#### Decision Tree

In [49]:

# Modelo base
Tree_classifier = DecisionTreeClassifier(max_depth=5,
                                         min_samples_leaf=10,
                                         min_samples_split=20, random_state=42)

# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", Tree_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="Decision_Tree_Classifier",
    parameters="Gini, max_depth=5, min_samples_leaf=10, min_samples_split=20, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", Tree_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="Decision_Tree_Classifier",
    parameters="Gini, max_depth=5, min_samples_leaf=10, min_samples_split=20, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", Tree_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="Decision_Tree_Classifier",
    parameters="Gini, max_depth=5, min_samples_leaf=10, min_samples_split=20, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_tree = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_tree

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Decision_Tree_Classifier,"Gini, max_depth=5, min_samples_leaf=10, min_sa...",0.4314 ± 0.0352,0.5522 ± 0.0260,0.3968 ± 0.0313,0.5248 ± 0.0229,0.3829 ± 0.0306,0.5211 ± 0.0264,0.4077 ± 0.0302,0.5371 ± 0.0255,0.4194 ± 0.0559,0.5760 ± 0.0350,0.3968 ± 0.0313,0.5248 ± 0.0229,0.6735 ± 0.0155,0.7435 ± 0.0151,0.5167 ± 0.0259,0.6246 ± 0.0195
1,Decision_Tree_Classifier,"Gini, max_depth=5, min_samples_leaf=10, min_sa...",0.4314 ± 0.0352,0.5522 ± 0.0260,0.3968 ± 0.0313,0.5248 ± 0.0229,0.3829 ± 0.0306,0.5211 ± 0.0264,0.4077 ± 0.0302,0.5371 ± 0.0255,0.4194 ± 0.0559,0.5760 ± 0.0350,0.3968 ± 0.0313,0.5248 ± 0.0229,0.6735 ± 0.0155,0.7435 ± 0.0151,0.5167 ± 0.0259,0.6246 ± 0.0195
2,Decision_Tree_Classifier,"Gini, max_depth=5, min_samples_leaf=10, min_sa...",0.4314 ± 0.0352,0.5522 ± 0.0260,0.3968 ± 0.0313,0.5248 ± 0.0229,0.3829 ± 0.0306,0.5211 ± 0.0264,0.4077 ± 0.0302,0.5371 ± 0.0255,0.4194 ± 0.0559,0.5760 ± 0.0350,0.3968 ± 0.0313,0.5248 ± 0.0229,0.6735 ± 0.0155,0.7435 ± 0.0151,0.5167 ± 0.0259,0.6246 ± 0.0195


#### Random Forest

In [ ]:

# Modelo base
RF_classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,           
    min_samples_leaf=5,     
    bootstrap=True,
    random_state=42,
)


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", RF_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="Random_Forest_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RF_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="Random_Forest_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", RF_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="Random_Forest_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_rf = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_rf

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Random_Forest_Classifier,"Default, No Scaler (Pipeline)",0.4244 ± 0.0322,0.7256 ± 0.0101,0.3888 ± 0.0322,0.6943 ± 0.0110,0.3741 ± 0.0366,0.7019 ± 0.0134,0.4007 ± 0.0337,0.7160 ± 0.0110,0.4076 ± 0.0669,0.7568 ± 0.0206,0.3888 ± 0.0322,0.6943 ± 0.0110,0.6688 ± 0.0162,0.8381 ± 0.0048,0.5097 ± 0.0265,0.7628 ± 0.0080
1,Random_Forest_Classifier,"Default, StandardScaler (Pipeline)",0.4258 ± 0.0339,0.7266 ± 0.0104,0.3901 ± 0.0333,0.6957 ± 0.0118,0.3750 ± 0.0379,0.7035 ± 0.0143,0.4015 ± 0.0355,0.7173 ± 0.0116,0.4090 ± 0.0670,0.7576 ± 0.0207,0.3901 ± 0.0333,0.6957 ± 0.0118,0.6689 ± 0.0174,0.8387 ± 0.0049,0.5106 ± 0.0277,0.7638 ± 0.0084
2,Random_Forest_Classifier,"Default, MinMaxScaler (Pipeline)",0.4258 ± 0.0315,0.7253 ± 0.0089,0.3901 ± 0.0312,0.6941 ± 0.0099,0.3752 ± 0.0357,0.7017 ± 0.0123,0.4020 ± 0.0329,0.7158 ± 0.0098,0.4082 ± 0.0660,0.7555 ± 0.0192,0.3901 ± 0.0312,0.6941 ± 0.0099,0.6697 ± 0.0159,0.8381 ± 0.0043,0.5109 ± 0.0257,0.7627 ± 0.0070


#### Extra Trees 

#### Gradient Boosting

#### XGBoost

#### LightGBM

#### CarBoost